# Isolation Forest Anomaly Detector

This notebook introduces a machine-learning approach to detecting anomalies in the CATS telemetry dataset using Isolation Forest.

Unlike the rule-based and rolling statistical detectors, which use predefined thresholds, Isolation Forest learns patterns from the input data and assigns anomaly scores based on how easily observations can be isolated.

The model will use the 17 telemetry signals as input features. The ground-truth label `y`, the event `category`, and event metadata will be excluded from the model features. These fields will be reserved for evaluation.

The first 1,000,000 observations form the nominal period, while the remaining 4,000,000 observations form the evaluation period. We will use the nominal period to establish the model-fitting strategy and evaluate predictions on the later period without using its ground-truth labels to generate predictions.

The model will be evaluated using the same observation-level metrics as the earlier detectors, including precision, recall, and false-alarm rate. Its performance will also be assessed at the event level.

The objective is to determine whether Isolation Forest improves anomaly detection compared with the rule-based and rolling statistical baselines.

## Step 2 — Verify Dataset Files and Required Packages

This step verifies the locations of the CATS telemetry dataset and its event metadata, and checks whether the required machine-learning and data-processing packages are installed.

We will inspect the Parquet file's metadata to confirm its row count and columns without loading the complete dataset.

This verification helps establish that the files and dependencies are available before we prepare the model inputs. No model fitting or prediction will occur in this step.

In [3]:

from pathlib import Path
from importlib.metadata import version

import pyarrow.parquet as pq

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "cats" / "data.parquet"
METADATA_PATH = PROJECT_ROOT / "data" / "raw" / "metadata.csv"

# print("Project root:", PROJECT_ROOT)
print("Telemetry file exists:", DATA_PATH.exists())
print("Metadata file exists:", METADATA_PATH.exists())

if DATA_PATH.exists():
    parquet_file = pq.ParquetFile(DATA_PATH)

    print("Dataset rows:", parquet_file.metadata.num_rows)
    print("Dataset columns:", parquet_file.schema_arrow.names)
    print("Telemetry file size (GB):",
          round(DATA_PATH.stat().st_size / (1024**3), 2))

if METADATA_PATH.exists():
    print("Metadata file size (MB):",
          round(METADATA_PATH.stat().st_size / (1024**2), 2))

for package in ["numpy", "pandas", "scikit-learn", "pyarrow"]:
    try:
        print(f"{package} version:", version(package))
    except Exception:
        print(f"{package}: NOT INSTALLED")


Telemetry file exists: True
Metadata file exists: True
Dataset rows: 5000000
Dataset columns: ['aimp', 'amud', 'arnd', 'asin1', 'asin2', 'adbr', 'adfl', 'bed1', 'bed2', 'bfo1', 'bfo2', 'bso1', 'bso2', 'bso3', 'ced1', 'cfo1', 'cso1', 'y', 'category', '__index_level_0__']
Telemetry file size (GB): 0.52
Metadata file size (MB): 0.01
numpy version: 2.4.6
pandas version: 3.0.6
scikit-learn version: 1.9.1
pyarrow version: 25.0.1


### Conclusion

The CATS telemetry file and event metadata are present at the expected project paths. The Parquet metadata confirms that the dataset contains 5,000,000 observations, and all required packages are installed.

The telemetry file includes 17 signals, the ground-truth label `y`, the event category, and the stored index column. The timestamp index must be preserved during data loading, while only the 17 telemetry signals will be used as model features.

The local verification is complete. The next step is to prepare the Kaggle execution environment, where we will have a better opportunity to process the dataset without exhausting the laptop's available memory.

## Step 3 — Load the Dataset and Prepare Model Inputs

We will load the CATS telemetry dataset from the Kaggle input directory and separate the 17 telemetry signals from the ground-truth labels and event categories.

The first 1,000,000 observations will be used as the nominal training period, and the remaining 4,000,000 observations will be reserved for evaluation.

The Isolation Forest model will use only the telemetry signals. The ground-truth label `y` and event category will be retained for evaluation but excluded from model training.

We will preserve the timestamp index to support event-level evaluation and detection-delay analysis.

In [ ]:
from pathlib import Path

INPUT_ROOT = Path(
    "/kaggle/input/datasets/patrickfleith/"
    "controlled-anomalies-time-series-dataset"
)

for path in INPUT_ROOT.rglob("*"):
    if path.is_file():
        print(path)

## Step 3B — Load the CATS Dataset

The mounted Kaggle dataset provides the telemetry data as `data.csv` and the event metadata as `metadata.csv`.

We will load the telemetry data, preserve its time index, verify the required columns, and separate the first 1,000,000 observations from the remaining 4,000,000 observations. Only the 17 telemetry signals will be used as model features; labels and categories will be reserved for evaluation.

In [ ]:
from pathlib import Path

import pandas as pd

INPUT_ROOT = Path(
    "/kaggle/input/datasets/patrickfleith/"
    "controlled-anomalies-time-series-dataset"
)

DATA_PATH = INPUT_ROOT / "data.csv"
METADATA_PATH = INPUT_ROOT / "metadata.csv"

signal_columns = [
    "aimp", "amud", "arnd", "asin1", "asin2",
    "adbr", "adfl", "bed1", "bed2", "bfo1",
    "bfo2", "bso1", "bso2", "bso3", "ced1",
    "cfo1", "cso1"
]

# Read the telemetry CSV.
df = pd.read_csv(DATA_PATH, index_col=0, parse_dates=True)
event_metadata = pd.read_csv(METADATA_PATH)

required_columns = signal_columns + ["y", "category"]
missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

if len(df) != 5_000_000:
    raise ValueError(f"Unexpected row count: {len(df):,}")

X_train = df.iloc[:1_000_000][signal_columns]
X_eval = df.iloc[1_000_000:][signal_columns]

y_eval = df.iloc[1_000_000:]["y"].astype("int8")
category_eval = df.iloc[1_000_000:]["category"]

print("Dataset shape:", df.shape)
print("Training shape:", X_train.shape)
print("Evaluation shape:", X_eval.shape)
print("Index type:", type(df.index).__name__)
print("First timestamp:", df.index.min())
print("Last timestamp:", df.index.max())
print("Event metadata shape:", event_metadata.shape)
print("Missing training values:", int(X_train.isna().sum().sum()))
print("Missing evaluation values:", int(X_eval.isna().sum().sum()))
print("Training label counts:")
print(df.iloc[:1_000_000]["y"].value_counts(dropna=False))

### Conclusion

The CATS dataset loaded successfully in Kaggle with 5,000,000 observations, 17 telemetry signals, a preserved datetime index, and 200 event records in the metadata.

The first 1,000,000 observations contain only normal labels, and neither the training nor evaluation telemetry contains missing values. The training and evaluation feature matrices have the expected dimensions.

The data is ready for the Isolation Forest fitting strategy. The model will be fitted using normal-period telemetry features, while the later four million observations and their ground-truth labels will remain reserved for evaluation.

## Step 4 — Configure Isolation Forest

Isolation Forest is an unsupervised anomaly-detection algorithm that identifies unusual observations by recursively partitioning the feature space. Anomalies tend to be isolated in fewer splits than observations that follow common patterns.

### 4.1 Mathematical foundation of Isolation Forest

**Average path length**

Let \(h_t(x)\) represent the number of splits required to isolate observation \(x\) in tree \(t\). For a forest containing \(T\) trees, the average path length is

$$
E[h(x)] = \frac{1}{T}\sum_{t=1}^{T}h_t(x)
$$

where:

- \(x\) is an observation represented by the 17 telemetry signals.
- \(h_t(x)\) is its path length in tree \(t\).
- \(T\) is the total number of trees.
- \(E[h(x)]\) is the average path length across the forest.

An observation that is isolated quickly generally has a shorter average path length and is more likely to be anomalous.

**Anomaly score**

Isolation Forest normalizes the average path length to calculate an anomaly score:

$$
s(x,n)=2^{-\frac{E[h(x)]}{c(n)}}
$$

where \(n\) is the number of observations used to build each tree, and \(c(n)\) is the expected path length used for normalization.

For \(n>2\), the normalization term is

$$
c(n)=2H(n-1)-\frac{2(n-1)}{n}
$$

Here, \(H(n-1)\) is the harmonic number

$$
H(n-1)=\sum_{k=1}^{n-1}\frac{1}{k}
$$

For large \(n\), the harmonic number can be approximated by

$$
H(n-1)\approx \ln(n-1)+\gamma
$$

where \(\ln\) is the natural logarithm and \(\gamma\approx0.57721\) is the Euler–Mascheroni constant.

The score generally ranges between 0 and 1:

- Scores closer to 1 indicate observations that are easier to isolate.
- Scores around 0.5 are near the theoretical reference point of the normalized score.
- Lower scores generally indicate observations that require longer paths to isolate.

These interpretations describe the scoring mechanism; the appropriate decision threshold must still be determined for our anomaly-detection task.

### 4.2 Training data and model parameters

The first 1,000,000 observations form our nominal training period. All of these observations have ground-truth label \(y=0\). We will use the 17 telemetry signals as model features and exclude the ground-truth labels and event categories from training.

We will configure two principal parameters:

- **`n_estimators`**: the number of isolation trees. More trees can stabilize anomaly scores, but increase computation.
- **`max_samples`**: the maximum number of observations used to build each tree. This controls the sample size per tree, even when the model is fitted using the full training matrix.

Our initial configuration is

$$
N=1{,}000{,}000,\qquad T=300,\qquad m=2{,}048
$$

where \(N\) is the total number of nominal training observations, \(T\) is the number of trees, and \(m\) is the maximum number of observations used by each tree.

The maximum number of training-row selections across all trees is therefore

$$
T\times m=300\times2{,}048=614{,}400
$$

This is a count of selections across trees, not necessarily 614,400 unique observations, because different trees can use overlapping subsets. The model receives the full training matrix, but each tree uses its own subsample.

We will begin with 300 trees and 2,048 observations per tree as an initial configuration, not as a proven optimum. We will assess the resulting model and, if warranted, compare it with configurations that use more observations per tree or additional trees.

### 4.3 Evaluation and leakage prevention

The later 4,000,000 observations will remain reserved for evaluation. Their labels and event categories will not be used to fit the model or tune its decision threshold.

We will evaluate observation-level precision, recall, and false-alarm rate, as well as event-level detection and detection delay. We will also record training and prediction time so that performance can be assessed alongside computational cost.

The objective is not simply to maximize the number of training observations. It is to determine which model configuration provides reliable anomaly detection on unseen observations without using evaluation labels to influence model selection.

In [ ]:
from sklearn.ensemble import IsolationForest

isolation_forest = IsolationForest(
    n_estimators=300,
    max_samples=2048,
    contamination="auto",
    max_features=1.0,
    bootstrap=False,
    n_jobs=-1,
    random_state=42
)

print("Training observations:", len(X_train))
print("Features:", X_train.shape[1])
print("Trees:", isolation_forest.n_estimators)
print("Maximum samples per tree:", isolation_forest.max_samples)
print("Parallel CPU workers:", isolation_forest.n_jobs)

### Conclusion

The Isolation Forest model is configured with 300 trees and a maximum of 2,048 observations per tree. It will use all 1,000,000 nominal training observations as its input matrix and all 17 telemetry signals as features.

The configuration is ready for fitting, but the current `n_jobs=1` setting limits parallel execution to one worker. Before training, we will change `n_jobs` to `-1` so the estimator can use the available CPU workers.

The next step is to fit the model and measure its actual training time. We will use the observed results to guide any subsequent changes to the configuration.

## Step 5 — Fit the Isolation Forest Model

We will fit Isolation Forest using the complete nominal training matrix, \(X_{\text{train}}\), containing 1,000,000 observations and 17 telemetry features.

During fitting, the algorithm constructs 300 isolation trees. Each tree uses a subsample of up to 2,048 observations, while the forest is fitted using the full training matrix.

The model does not use the ground-truth labels or event categories during fitting. This preserves the unsupervised learning setup.

We will measure the training time to assess the computational cost of this configuration. No evaluation predictions or threshold selection will occur in this step.

In [ ]:
import time

start_time = time.perf_counter()

isolation_forest.fit(X_train)

training_seconds = time.perf_counter() - start_time

print("Model fitted successfully.")
print("Training observations:", len(X_train))
print("Number of trees:", len(isolation_forest.estimators_))
print("Training time (seconds):", round(training_seconds, 2))
print("Training time (minutes):", round(training_seconds / 60, 2))

### Conclusion

Isolation Forest fitted successfully using the full 1,000,000-observation nominal training matrix and all 17 telemetry features. The forest contains the expected 300 trees and completed training in 4.55 seconds.

The measured fitting time indicates that this initial configuration is computationally manageable. However, training speed alone does not establish model quality. Each tree uses a subsample of up to 2,048 observations, so the configuration must be assessed using anomaly-detection metrics and event-level evaluation.

The fitted model is ready for anomaly scoring on the later evaluation period. Ground-truth labels and event categories will remain separate from the prediction process.

## Step 6 — Generate Isolation Forest Anomaly Scores

Isolation Forest assigns an anomaly score to each observation based on how readily the observation can be isolated by the trees.

In scikit-learn, `score_samples(X)` returns an opposite-anomaly score: **lower values indicate more anomalous observations**.

To make the direction easier to interpret, we will reverse the sign:

$$
A(x)=-\operatorname{score\_samples}(x)
$$

where \(A(x)\) is the resulting anomaly score. Higher values will then indicate observations that the model considers more anomalous.

We will calculate scores for all 4,000,000 evaluation observations. The evaluation labels and event categories will not be passed to the scoring method.

At this stage, we will generate scores only. We will not yet classify observations as anomalous or select a decision threshold.

In [ ]:
import time

start_time = time.perf_counter()

evaluation_scores = -isolation_forest.score_samples(X_eval)

scoring_seconds = time.perf_counter() - start_time

print("Evaluation observations scored:", len(evaluation_scores))
print("Score count matches evaluation rows:", len(evaluation_scores) == len(X_eval))
print("Scoring time (seconds):", round(scoring_seconds, 2))
print("Scoring time (minutes):", round(scoring_seconds / 60, 2))
print("Minimum anomaly score:", round(evaluation_scores.min(), 6))
print("Maximum anomaly score:", round(evaluation_scores.max(), 6))
print("Mean anomaly score:", round(evaluation_scores.mean(), 6))
print("Median anomaly score:", round(float(pd.Series(evaluation_scores).median()), 6))

### Conclusion

Isolation Forest successfully scored all 4,000,000 observations in the evaluation period in 55.68 seconds. The resulting anomaly scores range from 0.391042 to 0.720911, with a mean of 0.469028 and a median of 0.457208.

Because the scores were defined as the negative of scikit-learn's `score_samples` output, higher values indicate greater anomaly suspicion.

These summary statistics establish the range and central tendency of the scores, but they do not establish detection quality or determine an appropriate decision threshold. The next step is to examine the score distribution and define a threshold-selection strategy that does not use evaluation labels to tune the model.

## Step 7 — Examine the Anomaly-Score Distribution

Isolation Forest produces a continuous anomaly score rather than an immediate normal/anomaly classification.

Let the anomaly score for observation \(i\) be \(A_i\). A binary prediction can be created using a decision threshold \(\tau\):

$$
\hat{y}_i =
\begin{cases}
1, & \text{if } A_i > \tau \\
0, & \text{if } A_i \leq \tau
\end{cases}
$$

Here:

- **Aᵢ** is the anomaly score for observation *i*.
- **τ** is the selected decision threshold.
- **ŷᵢ = 1** means that observation *i* is classified as an anomaly.
- **ŷᵢ = 0** means that observation *i* is classified as normal.

The threshold τ determines which observations are classified as anomalies. A lower threshold will generally classify more observations as anomalous, while a higher threshold will classify fewer observations as anomalous.

Before selecting τ, we will examine the empirical distribution of the evaluation scores using quantiles. This allows us to understand how the scores are distributed without assuming that a particular numerical score represents an anomaly.

For a quantile \(q_p\), approximately \(p\) proportion of observations have scores less than or equal to \(q_p\). Therefore, the proportion above that quantile is approximately:

$$
1 - p
$$

For example, approximately \(1 - 0.99 = 0.01\), or 1%, of observations lie above the 99th percentile.

The ground-truth labels will not be used to calculate these score quantiles.

In [ ]:
score_series = pd.Series(evaluation_scores)

score_quantiles = score_series.quantile(
    [0.50, 0.75, 0.90, 0.95, 0.975, 0.99, 0.995, 0.999]
)

print("Isolation Forest anomaly-score quantiles:")
print(score_quantiles)

### Observation

The Isolation Forest anomaly scores increase progressively toward the upper tail. The median score is **0.4572**, while the 99th percentile is **0.5945** and the 99.9th percentile is **0.6413**.

This indicates that the model assigns relatively higher anomaly scores to a smaller proportion of observations.

However, these evaluation-score quantiles will be treated as **descriptive information only**. They will not be used to select the final anomaly threshold because the evaluation period contains the ground-truth anomaly labels used later for performance assessment.

The decision threshold should instead be established from the **nominal training period**, where all 1,000,000 observations are known to be normal. This preserves the separation between threshold selection and final evaluation.

### Conclusion

The Isolation Forest produces a continuous anomaly-score distribution with a distinct upper tail. This gives us a suitable score on which to define an anomaly decision rule.

We will now establish the threshold using scores from the **nominal training data**, rather than using the evaluation labels or evaluation performance. The resulting threshold will then be applied unchanged to the 4,000,000 evaluation observations.

In [ ]:

training_scores = -isolation_forest.score_samples(X_train)

training_score_series = pd.Series(training_scores)

training_score_quantiles = training_score_series.quantile(
    [0.95, 0.99, 0.995, 0.999]
)

print("Isolation Forest training-score quantiles:")
print(training_score_quantiles)


### Observation

The Isolation Forest anomaly-score quantiles increase from 0.531786 at the 95th percentile to 0.626215 at the 99.9th percentile. This shows the upper tail of the score distribution for the nominal training data.

The 99th-percentile score is approximately 0.577973. Using this value as a candidate threshold would flag approximately the highest-scoring 1% of nominal training observations. These observations are not necessarily actual anomalies; the threshold only describes how unusual their scores are relative to the nominal training distribution.

### Conclusion

The nominal training-score distribution provides a threshold-selection method that does not use evaluation labels. We will use the 99th percentile, approximately 0.577973, as an initial candidate threshold and evaluate its performance on the separate evaluation period.

The threshold will remain fixed during evaluation so that precision, recall, and false-alarm rate can be measured without tuning the threshold against the evaluation labels.

In [ ]:
threshold = training_score_quantiles.loc[0.99]

y_pred_if = (evaluation_scores > threshold).astype("int8")

print("Threshold:", round(float(threshold), 6))
print("Evaluation observations:", len(y_pred_if))
print("Predicted anomalies:", int(y_pred_if.sum()))
print("Predicted normal observations:", int((y_pred_if == 0).sum()))
print("Predicted anomaly rate:", round(float(y_pred_if.mean()), 4))

### Observation

The fixed threshold of 0.577973 classified 92,807 of the 4,000,000 evaluation observations as anomalies, giving a predicted anomaly rate of 2.32%. The remaining 3,907,193 observations were classified as normal.

The evaluation anomaly rate is higher than the approximately 1% of nominal training observations expected to exceed the training 99th-percentile threshold. This difference reflects the evaluation score distribution and does not, by itself, establish detection quality.

### Conclusion

The Isolation Forest has generated predictions for the complete evaluation period using a threshold derived from nominal training data. The predictions are ready for comparison with the ground-truth labels to determine precision, recall, and false-alarm rate.

In [ ]:

from sklearn.metrics import confusion_matrix, precision_score, recall_score

cm_if = confusion_matrix(y_eval, y_pred_if, labels=[0, 1])
tn, fp, fn, tp = cm_if.ravel()

precision_if = precision_score(y_eval, y_pred_if, zero_division=0)
recall_if = recall_score(y_eval, y_pred_if, zero_division=0)
far_if = fp / (fp + tn)

print("Confusion matrix:")
print(cm_if)

print("\nIsolation Forest evaluation:")
print("True negatives:", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives:", tp)
print("Precision:", round(precision_if, 4))
print("Recall:", round(recall_if, 4))
print("False-alarm rate:", round(far_if, 4))


### Observation

At the 99th-percentile nominal training threshold of 0.577973, Isolation Forest achieved 16.79% precision, 8.20% recall, and a 2.03% false-alarm rate on the evaluation period.

Compared with the rule-based and rolling statistical detectors, Isolation Forest produced the highest precision and lowest false-alarm rate, but substantially lower recall. It detected 15,581 of the 190,100 anomalous observations and missed 174,519.

### Conclusion

Isolation Forest provides a more conservative set of anomaly predictions at this threshold. Its low false-alarm rate may be useful when false alerts are costly, but its low recall is a major limitation when the objective is to detect as many anomalous observations as possible.

These results apply to the selected threshold, not every possible Isolation Forest configuration. We will retain the current results for comparison and proceed to event-level evaluation before deciding how the model should be assessed overall.

In [ ]:

detected_events = 0
missed_events = 0
event_detection_fractions = []
detection_delays = []

for _, event in event_metadata.iterrows():
    start_time = pd.to_datetime(event["start_time"])
    end_time = pd.to_datetime(event["end_time"])

    event_mask = (X_eval.index >= start_time) & (X_eval.index <= end_time)
    event_predictions = y_pred_if[event_mask]

    if len(event_predictions) == 0:
        missed_events += 1
        event_detection_fractions.append(0.0)
        continue

    detected_positions = (event_predictions == 1).nonzero()[0]
    event_detection_fractions.append(
        float((event_predictions == 1).mean())
    )

    if len(detected_positions) > 0:
        detected_events += 1
        first_detection_time = X_eval.index[event_mask][detected_positions[0]]
        detection_delays.append(
            (first_detection_time - start_time).total_seconds()
        )
    else:
        missed_events += 1

total_events = len(event_metadata)

print("Total events:", total_events)
print("Detected events:", detected_events)
print("Missed events:", missed_events)
print("Event detection rate:", round(detected_events / total_events, 4))
print(
    "Mean event detection fraction:",
    round(float(pd.Series(event_detection_fractions).mean()), 4)
)
print(
    "Median detection delay (seconds):",
    round(float(pd.Series(detection_delays).median()), 2)
    if detection_delays else "No events detected"
)
print(
    "Mean detection delay (seconds):",
    round(float(pd.Series(detection_delays).mean()), 2)
    if detection_delays else "No events detected"
)


In [ ]:
eval_start = X_eval.index.min()
eval_end = X_eval.index.max()

event_starts = pd.to_datetime(event_metadata["start_time"])
event_ends = pd.to_datetime(event_metadata["end_time"])

starts_before_eval = event_starts < eval_start
ends_before_eval = event_ends < eval_start
starts_after_eval = event_starts > eval_end
ends_after_eval = event_ends > eval_end

overlaps_eval = (event_ends >= eval_start) & (event_starts <= eval_end)

print("Evaluation start:", eval_start)
print("Evaluation end:", eval_end)
print("Events starting before evaluation:", int(starts_before_eval.sum()))
print("Events ending before evaluation:", int(ends_before_eval.sum()))
print("Events starting after evaluation:", int(starts_after_eval.sum()))
print("Events ending after evaluation:", int(ends_after_eval.sum()))
print("Events overlapping evaluation:", int(overlaps_eval.sum()))
print(
    "Events crossing evaluation start:",
    int((starts_before_eval & (event_ends >= eval_start)).sum())
)


### Step 11 — Observation

Isolation Forest detected 178 of the 200 documented anomaly events and missed 22, resulting in an event detection rate of 89%.

The mean event detection fraction was 5.22%, indicating that, averaged across all 200 events, only a small proportion of observations within each event were flagged. The median detection delay was 89.5 seconds, and the mean was 157.08 seconds. These delay statistics apply only to the 178 detected events.

Event detection rate differs from observation-level recall. An event is counted as detected if at least one observation within its interval is flagged, even when most anomalous observations are missed.

### Step 11 — Conclusion

Isolation Forest detected at least one anomalous observation in most documented events, but its low mean event detection fraction shows that it captured only a small proportion of event observations on average. The detection delays also indicate that identifying an event at least once does not guarantee prompt detection.

These results should be considered alongside the observation-level metrics from Step 10 when assessing the detector's practical effectiveness.

## Step 12 — Compare Isolation Forest with Existing Baselines

We have evaluated three anomaly detectors on the same 4,000,000-observation evaluation period: the rule-based detector, the rolling statistical detector, and Isolation Forest.

This comparison focuses on three observation-level metrics:

- **Precision:** the proportion of predicted anomalies that were actual anomalies.
- **Recall:** the proportion of actual anomalous observations detected.
- **False-alarm rate:** the proportion of actual normal observations incorrectly flagged.

The metrics are defined as follows:

$$
\text{Precision}=\frac{TP}{TP+FP}
$$

$$
\text{Recall}=\frac{TP}{TP+FN}
$$

$$
\text{FAR}=\frac{FP}{FP+TN}
$$

Here, \(TP\), \(FP\), \(TN\), and \(FN\) denote true positives, false positives, true negatives, and false negatives, respectively.

Comparing these metrics helps us understand the trade-off between detecting anomalies and avoiding false alarms. Event-level results provide additional information about whether each detector identifies documented events at least once.

In [ ]:

baseline_comparison = pd.DataFrame(
    {
        "Rule-based": {
            "Precision": 0.0985,
            "Recall": 0.5393,
            "False-alarm rate": 0.2463,
            "Event detection rate": 195 / 200,
        },
        "Rolling statistical": {
            "Precision": 0.0476,
            "Recall": 0.2640,
            "False-alarm rate": 0.2636,
            "Event detection rate": 200 / 200,
        },
        "Isolation Forest": {
            "Precision": precision_if,
            "Recall": recall_if,
            "False-alarm rate": far_if,
            "Event detection rate": detected_events / total_events,
        },
    }
).T

print("Detector performance comparison:")
display(baseline_comparison.round(4))


### Observation

The rule-based detector achieved the highest observation-level recall at 53.93%, while the rolling statistical detector detected at least one observation in all 200 documented events. Isolation Forest achieved the highest precision at 16.79% and the lowest false-alarm rate at 2.03%, but its recall was only 8.20% and it detected 178 of the 200 events.

These results demonstrate that detector performance depends on the metric being considered. Event detection rate measures whether an event was flagged at least once, whereas observation-level recall measures the proportion of all anomalous observations detected.

### Conclusion

No detector dominates across all the reported metrics. The rule-based detector identifies a larger proportion of anomalous observations but generates more false alarms. The rolling statistical detector achieves complete event coverage under the at-least-one-detection definition but also generates many false alarms. Isolation Forest is more conservative, producing fewer false alarms but missing many anomalies.

We will retain these results for the final model comparison. The next candidate is Local Outlier Factor (LOF), which uses local density differences to identify observations that are unusual relative to their neighbours.